# La forza centripeta — simulazione passo per passo (Colab)
**Fisica 3R — Liceo «L. da Vinci», Trento**

In Google Colab **VPython non funziona** (Colab usa un proprio sistema di notebook non compatibile con la grafica 3D di VPython). Qui facciamo la **stessa simulazione** con il metodo passo per passo e disegniamo i risultati con `matplotlib`.

Che cosa simuliamo: una pallina di massa $m$ parte con velocità $v_0$; su di lei agisce una forza di modulo costante $F$, **sempre perpendicolare alla velocità**. A ogni passo $\Delta t$:
$$a = \frac{F}{m},\qquad v_{n+1} = v_n + a\,\Delta t,\qquad x_{n+1} = x_n + v_{n+1}\,\Delta t$$

Esegui le celle una alla volta (▶ a sinistra, oppure Maiusc+Invio).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def simula(m, F, v0, dt, durata, t_taglio=None):
    """Metodo passo per passo. Restituisce tempi, posizioni x e y, modulo della velocità.
    Se t_taglio è un numero, da quell'istante la forza sparisce (il filo si taglia)."""
    x, y = 0.0, 0.0           # parte dall'origine...
    vx, vy = v0, 0.0          # ...verso destra
    T, X, Y, V = [0.0], [x], [y], [v0]
    t = 0.0
    while t < durata:
        v = np.hypot(vx, vy)
        if t_taglio is not None and t >= t_taglio:
            ax, ay = 0.0, 0.0                       # niente forza
        else:
            px, py = -vy / v, vx / v                # versore perpendicolare a v (verso sinistra)
            ax, ay = F / m * px, F / m * py         # seconda legge di Newton
        vx, vy = vx + ax * dt, vy + ay * dt         # nuova velocità
        x, y = x + vx * dt, y + vy * dt             # nuova posizione
        t += dt
        T.append(t); X.append(x); Y.append(y); V.append(np.hypot(vx, vy))
    return np.array(T), np.array(X), np.array(Y), np.array(V)

def misura(T, X, Y):
    """Raggio e periodo misurati sul primo giro."""
    for i in range(1, len(X)):
        if X[i-1] < 0 <= X[i] and T[i] > 0.1:          # tornata al punto di partenza
            return Y[:i+1].max() / 2, T[i]
    return Y.max() / 2, None

## 1. Che traiettoria si ottiene? Il modulo della velocità cambia?
Prima di eseguire, scrivi la tua previsione. Poi cambia i parametri e riesegui.

In [ ]:
m, F, v0 = 0.50, 2.0, 2.0     # kg, N, m/s
dt, durata = 0.001, 3.5        # s

T, X, Y, V = simula(m, F, v0, dt, durata)
r, Tp = misura(T, X, Y)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.5))
a1.plot(X, Y); a1.plot(0, 0, 'o'); a1.set_aspect('equal'); a1.grid(alpha=.3)
a1.set_title('Traiettoria'); a1.set_xlabel('x (m)'); a1.set_ylabel('y (m)')
a2.plot(T, V); a2.set_ylim(0, 1.5 * V.max()); a2.grid(alpha=.3)
a2.set_title('Modulo della velocità'); a2.set_xlabel('t (s)'); a2.set_ylabel('|v| (m/s)')
plt.show()
print(f"raggio r = {r:.3f} m    periodo T = {Tp:.3f} s    F*r/(m*v0^2) = {F*r/(m*v0**2):.3f}")

## 2. Il filo si taglia
A $t = 1{,}0\ \mathrm{s}$ la forza sparisce. Dove va la pallina? Perché?

In [ ]:
T, X, Y, V = simula(0.50, 2.0, 2.0, 0.001, 3.0, t_taglio=1.0)
plt.figure(figsize=(5.5, 5)); plt.plot(X, Y); plt.gca().set_aspect('equal'); plt.grid(alpha=.3)
plt.title('Il filo si taglia a t = 1,0 s'); plt.xlabel('x (m)'); plt.ylabel('y (m)'); plt.show()

## 3. Da che cosa dipende il raggio?
Teniamo fissi $m$ e $F$ e cambiamo $v_0$. Poi prova tu a cambiare $F$ e $m$ (una cosa alla volta).

In [ ]:
m, F = 0.50, 2.0
print(" v0 (m/s) |  r (m)  |  T (s)  | F*r/(m*v0^2)")
lista_v, lista_r = [], []
for v0 in [1.0, 2.0, 3.0, 4.0]:
    T, X, Y, V = simula(m, F, v0, 0.001, 2.2 * np.pi * m * v0 / F)
    r, Tp = misura(T, X, Y)
    lista_v.append(v0); lista_r.append(r)
    print(f"   {v0:4.1f}   |  {r:5.3f}  |  {Tp:5.3f}  |   {F*r/(m*v0**2):5.3f}")

v2 = np.array(lista_v)**2
plt.figure(figsize=(5.5, 4)); plt.plot(v2, lista_r, 'o-'); plt.grid(alpha=.3)
plt.xlabel('$v_0^2$ (m$^2$/s$^2$)'); plt.ylabel('r (m)'); plt.title('Il raggio in funzione di $v_0^2$'); plt.show()

**Domande.** Se $v_0$ raddoppia, come cambia $r$? E se raddoppia $F$? E se raddoppia $m$? Scrivi una formula che leghi $F$, $m$, $v_0$ e $r$.

## 4. Sfida: il passo temporale
Metti `dt = 0.1` nella cella del punto 1 e riesegui. Che cosa succede alla traiettoria e a $|v|$? È fisica o è un errore del metodo? Come lo correggi?